In [1]:

import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from pathlib import Path
import pandas as pd
import json

import time




/home/jet/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

device = "cuda:0"
dtype = torch.float16

model_id = "openai/whisper-large-v3"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, dtype=dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    dtype=dtype,
    device=device,
)


Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1259/1259 [00:00<00:00, 30154.00it/s]


In [3]:


# result = pipe(
#     "./Semantic Grounding Project/audio/segments/1-1_seg_34.30-34.70.wav",
#     generate_kwargs={
#         "task": "transcribe", 
#         "language": "en"
#     }
# )
# print(result["text"])

In [4]:
root_dir = Path("/home/jet/UCDavis/Gesture_Speech_Semantic_Grounding/Semantic Grounding Project/audio/segments")

In [5]:
audio_extensions = {".wav"}

In [6]:
audio_data = []
audio_suffix_set = set()



for audio_path in root_dir.rglob("*"):

 
    if audio_path.suffix.lower() in audio_extensions:

        # Assuming your structure is root_dir / category_folder / video.mp4
        # The parent folder name acts as the category/label
        category = audio_path.parent.name
        filename = audio_path.name
        relative_path = str(audio_path.relative_to(root_dir))
        absolute_path = str(audio_path.resolve())
    
        audio_data.append({
            "filename": filename,
            "category": category,
            "relative_path": relative_path,
            "absolute_path": absolute_path,
            "file_size_mb": round(audio_path.stat().st_size / (1024 * 1024), 2),
        })
#     else:
#         audio_suffix_set.add(audio_path)
#         print(f"{audio_path} extension not processed")
    
# print(audio_suffix_set)

In [7]:
audio_df = pd.DataFrame(audio_data)

In [8]:
print(f"Total audio files found: {len(audio_df)}")
display(audio_df.head())

# find . -type f | wc -l # same result

Total audio files found: 7157


,filename,category,relative_path,absolute_path,file_size_mb
0,2-3_seg_308.34-308.66.wav,segments,2-3_seg_308.34-308.66.wav,/home/jet/UCDavis/Gesture_Speech_Semantic_Grou...,0.17
1,4-6_seg_407.66-408.01.wav,segments,4-6_seg_407.66-408.01.wav,/home/jet/UCDavis/Gesture_Speech_Semantic_Grou...,0.34
2,3-2_seg_700.42-700.82.wav,segments,3-2_seg_700.42-700.82.wav,/home/jet/UCDavis/Gesture_Speech_Semantic_Grou...,0.17
3,2-2_seg_141.68-142.04.wav,segments,2-2_seg_141.68-142.04.wav,/home/jet/UCDavis/Gesture_Speech_Semantic_Grou...,0.34
4,4-2_seg_562.67-563.07.wav,segments,4-2_seg_562.67-563.07.wav,/home/jet/UCDavis/Gesture_Speech_Semantic_Grou...,0.34


In [23]:

file_to_transcription = []

num_files_to_process = 5

#list of all paths needed
absolute_paths = [row['absolute_path'] for index, row in audio_df.iterrows()][:num_files_to_process]
relative_paths = [row['relative_path'] for index, row in audio_df.iterrows()][:num_files_to_process]

start_time = time.perf_counter()
results = pipe(
    absolute_paths, 
    batch_size=8, 
    return_timestamps=True,
    generate_kwargs={
        "task": "transcribe", 
        "language": "en",
    }
)
end_time = time.perf_counter()
elapsed_time = end_time - start_time
print(f"Finished in {elapsed_time:.2f} seconds")
print(f"Average time of {elapsed_time / len(absolute_paths):.4f} seconds per file")
      
for rel_path, res in zip(relative_paths, results):
    temp_dict = {rel_path: res["text"]}
    file_to_transcription.append(temp_dict)


Finished in 1.37 seconds
Average time of 0.2731 seconds per file


In [28]:
print(f"{len(file_to_transcription)} transcriptions completed")

sorted_dict = sorted(file_to_transcription, key=lambda d: list(d.keys())[0])
# print(sorted_dict)

5 transcriptions completed
[{'2-2_seg_141.68-142.04.wav': ' would have to pick Malcolm Martin'}, {'2-3_seg_308.34-308.66.wav': ' mouth.'}, {'3-2_seg_700.42-700.82.wav': ' you actually just left.'}, {'4-2_seg_562.67-563.07.wav': " That's not what we're doing. That's not what it's about at all."}, {'4-6_seg_407.66-408.01.wav': ' who walked up naked through the town.'}]


In [29]:
with open("file_to_transcription.json", "w") as f:
    json.dump(sorted_dict, f, indent=4)

In [12]:
# Initial Results 10/30
# Finished in 793.65 seconds
# Average time of 0.2217 seconds per file
# potential error of 2–3% word error rate (WER) on clear audio or 7.4% WER on most english language benchmarks https://loronote.com/en/blog/how-accurate-is-whisper

In [ ]:
# 11/6 Results, sorted json array with 7157 files
# Total audio files found: 7157
# Finished in 1492.76 seconds
# Average time of 0.2086 seconds per file
# 7157 transcriptions completed